# MedMNIST v2 — Phase 3: Bias Audit, Fairness Analysis, Mitigation & Lightweight Models

**Trains 4 models in sequence — estimated ~5–6 hours on T4:**
1. Baseline ResNet-18 (standard CE + uniform sampler)
2. Weighted Sampler ResNet-18 (oversample minority classes)
3. Weighted Loss ResNet-18 (inverse-frequency CE weights)
4. Lightweight ResNet-18 (0.5× channel width)

**Requirements:** GPU T4 + Internet enabled.

## A — Setup

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU T4 in Kaggle settings'
print('GPU           :', torch.cuda.get_device_name(0))
print('VRAM          :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('PyTorch       :', torch.__version__)
torch.cuda.reset_peak_memory_stats()

In [ ]:
!pip install -q medmnist tensorboardX fire
print('Packages OK')

In [ ]:
import os, time, glob
from copy import deepcopy

os.chdir('/kaggle/working')
!git clone https://github.com/MedMNIST/experiments
os.makedirs('/kaggle/working/output/phase3', exist_ok=True)
print('Ready.')

## B — Imports, Model Definitions, Helpers

In [ ]:
import numpy as np
import pandas as pd
import PIL
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
from torchvision.models import resnet18
from medmnist import DermaMNIST
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_curve
)
from sklearn.preprocessing import label_binarize
from tqdm import trange

DEVICE     = torch.device('cuda:0')
N_CLASSES  = 7
NUM_EPOCHS = 100
BATCH      = 128
OUT        = '/kaggle/working/output/phase3'

CLASS_NAMES = [
    'Actinic keratoses', 'Basal cell carcinoma', 'Benign keratosis',
    'Dermatofibroma', 'Melanoma', 'Melanocytic nevi', 'Vascular lesions'
]
SHORT_NAMES = ['Actinic\nkerat.', 'Basal cell\ncarc.', 'Benign\nkerat.',
               'Dermato-\nfibroma', 'Melanoma', 'Melanocytic\nnevi', 'Vascular\nlesions']
COLORS = ['#e07b39','#4878d0','#6acc65','#d65f5f','#b47cc7','#c4ad66','#77bedb']

In [ ]:
# --- Lightweight ResNet-18 (0.5x channel width) ---
class _Block(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
        )
        self.skip = nn.Sequential(
            nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_c)
        ) if (stride != 1 or in_c != out_c) else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.net(x) + self.skip(x))

class SlimResNet18(nn.Module):
    """ResNet-18 with 0.5x channel width (32-32-64-128-256 vs 64-64-128-256-512)."""
    def __init__(self, num_classes=7, width_factor=0.5):
        super().__init__()
        c = int(64 * width_factor)
        self.stem = nn.Sequential(
            nn.Conv2d(3, c, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(c), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.layer1 = nn.Sequential(_Block(c,   c),         _Block(c,   c))
        self.layer2 = nn.Sequential(_Block(c,   c*2, 2),    _Block(c*2, c*2))
        self.layer3 = nn.Sequential(_Block(c*2, c*4, 2),    _Block(c*4, c*4))
        self.layer4 = nn.Sequential(_Block(c*4, c*8, 2),    _Block(c*8, c*8))
        self.head   = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                    nn.Linear(c*8, num_classes))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        return self.head(x)

# Quick sanity check
_slim = SlimResNet18(7)
_full = resnet18(weights=None, num_classes=7)
slim_params = sum(p.numel() for p in _slim.parameters())
full_params  = sum(p.numel() for p in _full.parameters())
print(f'SlimResNet18 params : {slim_params:,}  ({slim_params/full_params*100:.1f}% of full ResNet-18)')
print(f'Full ResNet-18 params: {full_params:,}')
del _slim, _full

In [ ]:
# --- Shared helper functions ---

def get_predictions(model, loader):
    model.eval()
    scores, targets = [], []
    with torch.no_grad():
        for x, y in loader:
            s = torch.softmax(model(x.to(DEVICE)), dim=1)
            scores.append(s.cpu().numpy())
            targets.append(y.numpy().flatten())
    return np.concatenate(targets), np.vstack(scores).argmax(1), np.vstack(scores)

def per_class_metrics(y_true, y_pred, y_score):
    y_bin = label_binarize(y_true, classes=list(range(N_CLASSES)))
    return pd.DataFrame({
        'Class':     CLASS_NAMES,
        'N':         np.bincount(y_true, minlength=N_CLASSES),
        'AUC':       [round(roc_auc_score(y_bin[:,i], y_score[:,i]), 4) for i in range(N_CLASSES)],
        'Accuracy':  np.round(confusion_matrix(y_true, y_pred).diagonal() /
                               confusion_matrix(y_true, y_pred).sum(axis=1), 4),
        'Precision': np.round(precision_score(y_true, y_pred, average=None, zero_division=0), 4),
        'Recall':    np.round(recall_score(y_true, y_pred, average=None, zero_division=0), 4),
        'F1':        np.round(f1_score(y_true, y_pred, average=None, zero_division=0), 4),
    })

def aggregate(y_true, y_pred, y_score):
    y_bin = label_binarize(y_true, classes=list(range(N_CLASSES)))
    return {
        'macro_auc': round(roc_auc_score(y_bin, y_score, average='macro'), 4),
        'accuracy':  round(accuracy_score(y_true, y_pred), 4),
        'macro_f1':  round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
    }

def run_experiment(name, model, train_loader, criterion, val_loader, test_loader):
    """Train model and return (best_model, y_true, y_pred, y_score, history_df)."""
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[int(0.5*NUM_EPOCHS), int(0.75*NUM_EPOCHS)], gamma=0.1)
    best_auc, best_state, history = 0, None, []
    t0 = time.time()

    for epoch in trange(NUM_EPOCHS, desc=name):
        model.train()
        losses = []
        for x, y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(x.to(DEVICE)), y.squeeze(1).long().to(DEVICE))
            loss.backward(); optimizer.step()
            losses.append(loss.item())
        scheduler.step()

        # Val AUC for early stopping (best model selection)
        vt, vp, vs = get_predictions(model, val_loader)
        y_bin = label_binarize(vt, classes=list(range(N_CLASSES)))
        val_auc = roc_auc_score(y_bin, vs, average='macro')
        history.append({'epoch': epoch, 'loss': np.mean(losses), 'val_auc': val_auc})
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = deepcopy(model.state_dict())

    elapsed = time.time() - t0
    model.load_state_dict(best_state)
    torch.save({'net': best_state}, f'{OUT}/best_{name}.pth')

    y_true, y_pred, y_score = get_predictions(model, test_loader)
    agg = aggregate(y_true, y_pred, y_score)
    print(f'\n{name}: AUC={agg["macro_auc"]:.4f}  Acc={agg["accuracy"]:.4f}  '
          f'F1={agg["macro_f1"]:.4f}  time={elapsed/60:.0f}min')
    return model, y_true, y_pred, y_score, pd.DataFrame(history)

print('Helpers defined.')

## C — Data Loading

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=PIL.Image.NEAREST),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5]),
])

ds_train = DermaMNIST(split='train', transform=transform, download=True)
ds_val   = DermaMNIST(split='val',   transform=transform, download=True)
ds_test  = DermaMNIST(split='test',  transform=transform, download=True)

train_labels = ds_train.labels.flatten()
class_counts = np.bincount(train_labels, minlength=N_CLASSES)

print('Train class distribution:')
for i, (n, c) in enumerate(zip(CLASS_NAMES, class_counts)):
    bar = '█' * int(c / class_counts.max() * 30)
    print(f'  {i} {n:<36}: {c:>5} ({c/len(train_labels)*100:4.1f}%) {bar}')
print(f'  Imbalance ratio (nevi/dermato): {class_counts[5]/class_counts[3]:.0f}:1')

# Standard loaders (used by baseline and lightweight)
loader_train    = DataLoader(ds_train, batch_size=BATCH, shuffle=True,  num_workers=2)
loader_train_e  = DataLoader(ds_train, batch_size=BATCH, shuffle=False, num_workers=2)
loader_val      = DataLoader(ds_val,   batch_size=BATCH, shuffle=False, num_workers=2)
loader_test     = DataLoader(ds_test,  batch_size=BATCH, shuffle=False, num_workers=2)

# Weighted sampler loader — oversample minority classes
sample_weights = (1.0 / class_counts)[train_labels]
sampler_ws     = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
loader_train_ws = DataLoader(ds_train, batch_size=BATCH, sampler=sampler_ws, num_workers=2)

# Weighted loss criterion — inverse frequency, normalized to mean=1
wl_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)
wl_weights = wl_weights / wl_weights.mean()
criterion_wl = nn.CrossEntropyLoss(weight=wl_weights.to(DEVICE))

# Standard criterion
criterion_std = nn.CrossEntropyLoss()

print('\nClass weights for weighted loss:')
for n, w in zip(CLASS_NAMES, wl_weights.numpy()):
    print(f'  {n:<36}: {w:.3f}')

## D — Training (4 models, ~5–6 hrs total)
Each model is saved to `/kaggle/working/output/phase3/` after training.

In [ ]:
# Model 1: Baseline ResNet-18
print('=== Training Model 1: Baseline ResNet-18 ===')
m_base = resnet18(weights=None, num_classes=N_CLASSES).to(DEVICE)
m_base, yt_base, yp_base, ys_base, hist_base = run_experiment(
    'baseline', m_base, loader_train, criterion_std, loader_val, loader_test)

In [ ]:
# Model 2: Weighted Sampler ResNet-18
print('=== Training Model 2: Weighted Sampler ResNet-18 ===')
m_ws = resnet18(weights=None, num_classes=N_CLASSES).to(DEVICE)
m_ws, yt_ws, yp_ws, ys_ws, hist_ws = run_experiment(
    'weighted_sampler', m_ws, loader_train_ws, criterion_std, loader_val, loader_test)

In [ ]:
# Model 3: Weighted Loss ResNet-18
print('=== Training Model 3: Weighted Loss ResNet-18 ===')
m_wl = resnet18(weights=None, num_classes=N_CLASSES).to(DEVICE)
m_wl, yt_wl, yp_wl, ys_wl, hist_wl = run_experiment(
    'weighted_loss', m_wl, loader_train, criterion_wl, loader_val, loader_test)

In [ ]:
# Model 4: Lightweight ResNet-18 (0.5x channels)
print('=== Training Model 4: Lightweight ResNet-18 (0.5x) ===')
m_slim = SlimResNet18(num_classes=N_CLASSES, width_factor=0.5).to(DEVICE)
m_slim, yt_slim, yp_slim, ys_slim, hist_slim = run_experiment(
    'lightweight', m_slim, loader_train, criterion_std, loader_val, loader_test)

## E — Bias Audit (Core Extension)

In [ ]:
# Per-class metrics for all 4 models
models_results = {
    'Baseline':         (yt_base, yp_base, ys_base),
    'Weighted Sampler': (yt_ws,   yp_ws,   ys_ws),
    'Weighted Loss':    (yt_wl,   yp_wl,   ys_wl),
    'Lightweight':      (yt_slim, yp_slim, ys_slim),
}

all_pc = {}
print('=== Per-Class Metrics — All Models ===')
for name, (yt, yp, ys) in models_results.items():
    df = per_class_metrics(yt, yp, ys)
    all_pc[name] = df
    agg = aggregate(yt, yp, ys)
    print(f'\n--- {name} (Macro AUC={agg["macro_auc"]:.4f}, Acc={agg["accuracy"]:.4f}, F1={agg["macro_f1"]:.4f}) ---')
    print(df[['Class','N','AUC','Accuracy','F1']].to_string(index=False))
    df.to_csv(f'{OUT}/perclass_{name.lower().replace(" ","_")}.csv', index=False)

In [ ]:
# Equity-accuracy tradeoff table: aggregate vs minority class performance
rows = []
for name, (yt, yp, ys) in models_results.items():
    agg = aggregate(yt, yp, ys)
    df  = per_class_metrics(yt, yp, ys)
    rows.append({
        'Model':           name,
        'Macro AUC':       agg['macro_auc'],
        'Accuracy':        agg['accuracy'],
        'Macro F1':        agg['macro_f1'],
        'Dermato F1':      float(df[df.Class=='Dermatofibroma']['F1'].values[0]),
        'Melanoma F1':     float(df[df.Class=='Melanoma']['F1'].values[0]),
        'Vascular F1':     float(df[df.Class=='Vascular lesions']['F1'].values[0]),
        'Nevi F1':         float(df[df.Class=='Melanocytic nevi']['F1'].values[0]),
        'Actinic F1':      float(df[df.Class=='Actinic keratoses']['F1'].values[0]),
    })

equity_df = pd.DataFrame(rows)
print('=== Equity-Accuracy Tradeoff Table ===')
print(equity_df.to_string(index=False))
equity_df.to_csv(f'{OUT}/equity_tradeoff.csv', index=False)
print('\nNotes:')
print('  Dermato F1, Vascular F1 = minority class fairness metrics')
print('  Melanoma F1 = clinically critical (misclassification = missed cancer)')
print('  Tradeoff: mitigation models should raise minority F1 but may drop Macro AUC')

In [ ]:
# Per-class ROC curves — baseline model, all 7 classes
y_bin = label_binarize(yt_base, classes=list(range(N_CLASSES)))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(N_CLASSES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], ys_base[:, i])
    auc_i = roc_auc_score(y_bin[:, i], ys_base[:, i])
    n_i   = int(np.bincount(yt_base, minlength=N_CLASSES)[i])
    axes[i].plot(fpr, tpr, color=COLORS[i], lw=2)
    axes[i].plot([0,1], [0,1], 'k--', lw=0.8)
    axes[i].set_title(f'{CLASS_NAMES[i]}\nAUC={auc_i:.3f}, N={n_i}', fontsize=8)
    axes[i].set_xlabel('FPR', fontsize=7); axes[i].set_ylabel('TPR', fontsize=7)
    axes[i].tick_params(labelsize=6)

axes[7].axis('off')
fig.suptitle('Per-Class ROC Curves — Baseline ResNet-18 (Test Set)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT}/roc_curves_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: roc_curves_baseline.png')

In [ ]:
# Confusion matrices — 2x2 grid for all 4 models
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
axes = axes.flatten()

for ax, (name, (yt, yp, _)) in zip(axes, models_results.items()):
    cm = confusion_matrix(yt, yp)
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(SHORT_NAMES, fontsize=6.5)
    ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(SHORT_NAMES, fontsize=6.5)
    ax.set_xlabel('Predicted', fontsize=8); ax.set_ylabel('True', fontsize=8)
    ax.set_title(f'{name}', fontsize=10)
    thresh = cm.max() / 2
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=6,
                    color='white' if cm[i,j] > thresh else 'black')

fig.suptitle('Confusion Matrices — All 4 Models (DermaMNIST Test Set)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUT}/confusion_matrices_all.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrices_all.png')

In [ ]:
# Class frequency vs per-class accuracy scatter (quantify the relationship)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
train_counts = np.bincount(ds_train.labels.flatten(), minlength=N_CLASSES)

for ax, (name, (yt, yp, ys)) in zip(axes, list(models_results.items())[:2]):
    cm = confusion_matrix(yt, yp)
    per_acc = cm.diagonal() / cm.sum(axis=1)
    y_bin = label_binarize(yt, classes=list(range(N_CLASSES)))
    per_auc = [roc_auc_score(y_bin[:,i], ys[:,i]) for i in range(N_CLASSES)]

    for i in range(N_CLASSES):
        ax.scatter(train_counts[i], per_acc[i], color=COLORS[i], s=120, zorder=5)
        ax.annotate(CLASS_NAMES[i].split()[0], (train_counts[i], per_acc[i]),
                    textcoords='offset points', xytext=(6, 3), fontsize=7)

    # Pearson correlation
    corr = np.corrcoef(train_counts, per_acc)[0, 1]
    ax.set_xlabel('Training samples per class', fontsize=9)
    ax.set_ylabel('Per-class accuracy (test)', fontsize=9)
    ax.set_title(f'{name}\nPearson r = {corr:.3f} (freq vs accuracy)', fontsize=9)
    ax.set_xscale('log')

plt.suptitle('Class Frequency vs Per-Class Accuracy', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT}/freq_vs_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

# Print Pearson r for all models
print('\nPearson r (train frequency vs per-class accuracy):')
for name, (yt, yp, _) in models_results.items():
    cm = confusion_matrix(yt, yp)
    per_acc = cm.diagonal() / cm.sum(axis=1)
    r = np.corrcoef(train_counts, per_acc)[0, 1]
    print(f'  {name:<25}: r = {r:.4f}')
print('  (r closer to 0 = bias mitigation reducing frequency-accuracy correlation)')

In [ ]:
# Per-class F1 comparison: all 4 models side by side
x = np.arange(N_CLASSES)
w = 0.2
fig, ax = plt.subplots(figsize=(14, 5))
offsets = [-1.5*w, -0.5*w, 0.5*w, 1.5*w]
model_colors = ['#4878d0', '#e07b39', '#6acc65', '#b47cc7']

for (name, (yt, yp, _)), offset, color in zip(models_results.items(), offsets, model_colors):
    f1s = f1_score(yt, yp, average=None, zero_division=0)
    ax.bar(x + offset, f1s, w, label=name, color=color, alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(SHORT_NAMES, fontsize=8)
ax.set_ylabel('F1 Score'); ax.set_ylim(0, 1)
ax.set_title('Per-Class F1: Baseline vs Mitigation vs Lightweight')
ax.legend(fontsize=8)
ax.axhline(0, color='black', lw=0.5)
plt.tight_layout()
plt.savefig(f'{OUT}/perclass_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: perclass_f1_comparison.png')

## F — Fairness & Demographic Analysis

In [ ]:
# Model confidence (max softmax prob) distribution per class — baseline model
# High confidence on majority class, low confidence on minority = uncertainty bias
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

max_conf = ys_base.max(axis=1)          # max softmax per sample
correct  = (yp_base == yt_base)         # correct prediction mask

conf_stats = []
for i in range(N_CLASSES):
    mask = yt_base == i
    conf_i = max_conf[mask]
    corr_i = correct[mask]
    n_i    = mask.sum()

    axes[i].hist(conf_i[corr_i],  bins=20, alpha=0.7, color='green',  label=f'Correct (n={corr_i.sum()})')
    axes[i].hist(conf_i[~corr_i], bins=20, alpha=0.7, color='red',    label=f'Wrong (n={(~corr_i).sum()})')
    axes[i].set_title(f'{CLASS_NAMES[i]}\nN={n_i}, mean conf={conf_i.mean():.2f}', fontsize=7.5)
    axes[i].set_xlabel('Max softmax probability', fontsize=7)
    axes[i].legend(fontsize=6)

    conf_stats.append({
        'Class':          CLASS_NAMES[i],
        'N':              int(n_i),
        'Mean conf':      round(float(conf_i.mean()), 4),
        'Mean conf (correct)':  round(float(conf_i[corr_i].mean()), 4) if corr_i.sum() > 0 else None,
        'Mean conf (wrong)':    round(float(conf_i[~corr_i].mean()), 4) if (~corr_i).sum() > 0 else None,
    })

axes[7].axis('off')
fig.suptitle('Model Confidence Distribution per Class\n'
             'HAM10000 source: primarily Austria/Australia, lighter skin tones\n'
             'Low confidence on minority classes proxies demographic underrepresentation',
             fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUT}/confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

conf_df = pd.DataFrame(conf_stats)
print('\n=== Confidence Statistics per Class ===')
print(conf_df.to_string(index=False))
conf_df.to_csv(f'{OUT}/confidence_stats.csv', index=False)
print('\nKey: classes with high wrong-prediction confidence = model is confidently wrong')
print('     classes with low mean confidence = model is uncertain = underrepresented in training')

In [ ]:
# Summary confidence box plots per class — cleaner view for paper
fig, ax = plt.subplots(figsize=(12, 5))

data_per_class = [max_conf[yt_base == i] for i in range(N_CLASSES)]
bp = ax.boxplot(data_per_class, patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color); patch.set_alpha(0.7)

ax.set_xticks(range(1, N_CLASSES+1))
ax.set_xticklabels(SHORT_NAMES, fontsize=8)
ax.set_ylabel('Max softmax probability (model confidence)')
ax.set_title('Prediction Confidence Distribution by Lesion Type\n'
             'Lower median = model more uncertain = proxy for underrepresentation in HAM10000')

# Annotate with training counts
train_c = np.bincount(ds_train.labels.flatten(), minlength=N_CLASSES)
for i in range(N_CLASSES):
    ax.text(i+1, 0.22, f'n={train_c[i]}', ha='center', fontsize=7, color='#555')

plt.tight_layout()
plt.savefig(f'{OUT}/confidence_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confidence_boxplots.png')

## G — Lightweight Model Analysis

In [ ]:
# Model size (MB) and inference time (ms/image)
import io, tempfile

def model_size_mb(model):
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.tell() / 1e6

def inference_time_ms(model, n_runs=200, batch_size=1):
    model.eval()
    x = torch.randn(batch_size, 3, 224, 224).to(DEVICE)
    # Warmup
    for _ in range(10):
        _ = model(x)
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_runs):
        _ = model(x)
    torch.cuda.synchronize()
    return (time.time() - t0) / n_runs * 1000  # ms per image

size_rows = []
for name, model in [('Baseline ResNet-18', m_base), ('Lightweight ResNet-18 (0.5x)', m_slim)]:
    params  = sum(p.numel() for p in model.parameters())
    size_mb = model_size_mb(model)
    inf_ms  = inference_time_ms(model)
    size_rows.append({'Model': name, 'Params': f'{params:,}', 'Size (MB)': round(size_mb, 1),
                      'Inference (ms/img)': round(inf_ms, 2)})

size_df = pd.DataFrame(size_rows)
print('=== Lightweight Model — Size & Speed ===')
print(size_df.to_string(index=False))
size_df.to_csv(f'{OUT}/model_size_inference.csv', index=False)

# AUC/Acc comparison
agg_base = aggregate(yt_base, yp_base, ys_base)
agg_slim = aggregate(yt_slim, yp_slim, ys_slim)
print(f'\nBaseline  : AUC={agg_base["macro_auc"]:.4f}, Acc={agg_base["accuracy"]:.4f}')
print(f'Lightweight: AUC={agg_slim["macro_auc"]:.4f}, Acc={agg_slim["accuracy"]:.4f}')
print(f'Delta      : AUC={agg_slim["macro_auc"]-agg_base["macro_auc"]:+.4f}, '
      f'Acc={agg_slim["accuracy"]-agg_base["accuracy"]:+.4f}')

## H — Robustness Testing (Optional Extension)

In [ ]:
# Corruption types: Gaussian noise, JPEG compression, brightness shift
# Tests whether minority classes degrade faster under distribution shift

import io as _io
from PIL import Image as _PIL

def corrupt_batch(x_batch, corruption):
    """Apply corruption to a batch tensor (B, 3, 224, 224) already normalized."""
    x = x_batch.clone()
    if corruption == 'gaussian':
        x = x + torch.randn_like(x) * 0.15
    elif corruption == 'brightness_dark':
        x = x * 0.5
    elif corruption == 'brightness_bright':
        x = x * 1.5
    return x

def evaluate_corrupted(model, loader, corruption):
    model.eval()
    all_scores, all_targets = [], []
    with torch.no_grad():
        for x, y in loader:
            xc = corrupt_batch(x.to(DEVICE), corruption)
            s  = torch.softmax(model(xc), dim=1)
            all_scores.append(s.cpu().numpy())
            all_targets.append(y.numpy().flatten())
    yt = np.concatenate(all_targets)
    ys = np.vstack(all_scores)
    yp = ys.argmax(1)
    return yt, yp, ys

corruptions = ['gaussian', 'brightness_dark', 'brightness_bright']
rob_rows = []

# Clean baseline
agg_clean = aggregate(yt_base, yp_base, ys_base)
rob_rows.append({'Model': 'Baseline', 'Corruption': 'Clean',
                 **agg_clean,
                 'Dermato F1':  float(f1_score(yt_base, yp_base, average=None, zero_division=0)[3]),
                 'Melanoma F1': float(f1_score(yt_base, yp_base, average=None, zero_division=0)[4]),
                 'Nevi F1':     float(f1_score(yt_base, yp_base, average=None, zero_division=0)[5])})

for corr in corruptions:
    yt_c, yp_c, ys_c = evaluate_corrupted(m_base, loader_test, corr)
    agg_c = aggregate(yt_c, yp_c, ys_c)
    f1s_c = f1_score(yt_c, yp_c, average=None, zero_division=0)
    rob_rows.append({'Model': 'Baseline', 'Corruption': corr,
                     **agg_c, 'Dermato F1': float(f1s_c[3]),
                     'Melanoma F1': float(f1s_c[4]), 'Nevi F1': float(f1s_c[5])})
    print(f'{corr:<22}: AUC={agg_c["macro_auc"]:.4f}  Acc={agg_c["accuracy"]:.4f}  '
          f'Dermato_F1={f1s_c[3]:.3f}  Melanoma_F1={f1s_c[4]:.3f}  Nevi_F1={f1s_c[5]:.3f}')

rob_df = pd.DataFrame(rob_rows)
rob_df = rob_df.round(4)
print('\n=== Robustness Table ===')
print(rob_df.to_string(index=False))
rob_df.to_csv(f'{OUT}/robustness_results.csv', index=False)
print('\nKey: if minority class F1 degrades faster than Nevi F1 under corruption,')
print('     model is less robust for under-represented groups (simulating low-quality cameras).')

## I — Final Summary + Save All Outputs

In [ ]:
print('=== PHASE 3 COMPLETE — OUTPUT FILES ===')
all_files = sorted(glob.glob(f'{OUT}/**/*', recursive=True) + glob.glob('/kaggle/working/*.csv'))
for f in all_files:
    if os.path.isfile(f):
        print(f'  {os.path.basename(f):<55} {os.path.getsize(f)/1024:>7.1f} KB')

print('\n=== EQUITY-ACCURACY TRADEOFF SUMMARY ===')
print(equity_df[['Model','Macro AUC','Accuracy','Dermato F1','Melanoma F1','Nevi F1']].to_string(index=False))

print('\n=== LIGHTWEIGHT MODEL SUMMARY ===')
print(size_df.to_string(index=False))

print('\n=== PEAK GPU MEMORY ===')
print(f'  {torch.cuda.max_memory_allocated()/1e9:.2f} GB allocated')
print(f'  {torch.cuda.max_memory_reserved()/1e9:.2f} GB reserved')